# 5.5 · Connecting real hosts: local and remote servers

**接入真实宿主：本地与远程服务器**

> **After this lesson you can:** Configure Claude Desktop and similar hosts to launch local stdio servers and connect remote ones, and diagnose the classic setup failures from logs.

Official reading: [Connect to local MCP servers](https://modelcontextprotocol.io/docs/2026-07-28/develop/connect-local-servers) · [Connect to remote MCP servers](https://modelcontextprotocol.io/docs/2026-07-28/develop/connect-remote-servers)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. A local server that behaves like the filesystem server
The official Filesystem Server takes the directories it may touch as **command-line arguments**. Ours does the same, and like many real servers it also needs a secret in an **environment variable**. It writes its startup messages to **stderr**, which is where a host's per-server log comes from. Everything lives in a temp folder.

In [ ]:
import asyncio
import json
import logging
import os
import platform
import shutil
import socket
import sys
import tempfile
import textwrap
import threading
import time
from contextlib import AsyncExitStack
from pathlib import Path

from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

for name in ("httpx", "httpx2", "mcp", "uvicorn"):
    logging.getLogger(name).setLevel(logging.WARNING)

work = Path(tempfile.mkdtemp())
(work / "Desktop").mkdir()
(work / "Downloads").mkdir()
(work / "Desktop" / "todo.md").write_text("- learn MCP\n- write a server\n")
(work / "Downloads" / "invoice.txt").write_text("Invoice #42: $120\n")

SERVER = work / "notes_server.py"
SERVER.write_text(textwrap.dedent('''
    import os
    import sys
    from pathlib import Path

    from mcp.server import MCPServer
    from mcp.server.mcpserver.exceptions import ToolError

    TOKEN = os.environ.get("NOTES_API_TOKEN")
    if not TOKEN:
        print("notes: NOTES_API_TOKEN is not set; refusing to start", file=sys.stderr)
        sys.exit(1)
    ROOTS = [Path(arg).resolve() for arg in sys.argv[1:]]
    for root in ROOTS:
        if not root.is_dir():
            print(f"notes: allowed directory {root} does not exist (cwd={os.getcwd()})", file=sys.stderr)
            sys.exit(1)
    print(f"notes: serving {len(ROOTS)} directories", file=sys.stderr)

    server = MCPServer("notes")


    def inside(path: str) -> Path:
        p = Path(path).resolve()
        if not any(p.is_relative_to(r) for r in ROOTS):
            raise ToolError(f"{path} is outside the allowed directories")
        return p


    @server.tool()
    def list_files() -> list[str]:
        """List every file in the allowed directories (absolute paths)."""
        return sorted(str(f) for r in ROOTS for f in r.rglob("*") if f.is_file())


    @server.tool()
    def read_file(path: str) -> str:
        """Read a text file inside an allowed directory."""
        return inside(path).read_text()


    @server.tool()
    def write_file(path: str, text: str) -> str:
        """Create or overwrite a text file inside an allowed directory."""
        inside(path).write_text(text)
        return f"wrote {len(text)} chars to {path}"


    @server.tool()
    def visible_env() -> list[str]:
        """Names (never values) of the environment variables this server process can see."""
        return sorted(os.environ)


    if __name__ == "__main__":
        server.run(transport="stdio")
'''))
print("server written to", SERVER)

### 2. Write and validate a `claude_desktop_config.json`
This is the exact shape Claude Desktop reads: a top-level `mcpServers` object whose keys are the friendly names shown in the UI and whose values say how to **launch** the server (`command`, `args`, optional `env`). Real locations, from the docs, are `~/Library/Application Support/Claude/claude_desktop_config.json` on macOS and `%APPDATA%\Claude\claude_desktop_config.json` on Windows. We write ours to the temp folder instead.

In [ ]:
config = {
    "mcpServers": {
        "notes": {
            "command": sys.executable,                      # absolute path to the interpreter
            "args": [str(SERVER), str(work / "Desktop"), str(work / "Downloads")],
            "env": {"NOTES_API_TOKEN": "demo-token-1234"},
        }
    }
}
CONFIG_PATH = work / "claude_desktop_config.json"
CONFIG_PATH.write_text(json.dumps(config, indent=2))

real_location = {"Darwin": "~/Library/Application Support/Claude/claude_desktop_config.json",
                 "Windows": r"%APPDATA%\Claude\claude_desktop_config.json"}
print("on this OS Claude Desktop would read:",
      real_location.get(platform.system(), "(not listed in the docs for this OS; use Settings → Developer → Edit Config)"))
print(CONFIG_PATH.read_text())

A host can't tell you much when a server never starts, so check the config before you restart the app. This validator encodes the troubleshooting checklist from the docs: valid JSON, the `mcpServers` key, absolute paths, a command that exists, and string-only `env` values.

In [ ]:
def validate_config(text: str) -> list[str]:
    try:
        data = json.loads(text)
    except json.JSONDecodeError as exc:
        return [f"invalid JSON at line {exc.lineno}, column {exc.colno}: {exc.msg}"]
    servers = data.get("mcpServers")
    if not isinstance(servers, dict):
        return ["top-level `mcpServers` object is missing"]
    problems = []
    for name, entry in servers.items():
        cmd = entry.get("command")
        if not isinstance(cmd, str) or not cmd:
            problems.append(f"{name}: `command` must be a non-empty string")
        elif not (Path(cmd).is_absolute() and Path(cmd).exists()) and shutil.which(cmd) is None:
            problems.append(f"{name}: command {cmd!r} not found on PATH (try an absolute path)")
        args = entry.get("args", [])
        if not isinstance(args, list) or not all(isinstance(a, str) for a in args):
            problems.append(f"{name}: `args` must be a list of strings")
        else:
            for a in args:
                looks_like_path = "/" in a or "\\" in a or a.startswith(".")
                if looks_like_path and not Path(a).is_absolute():
                    problems.append(f"{name}: relative path {a!r} (the host's working directory is undefined)")
        env = entry.get("env", {})
        if not isinstance(env, dict) or not all(isinstance(v, str) for v in env.values()):
            problems.append(f"{name}: `env` must map names to string values")
    return problems


print("our config:", validate_config(CONFIG_PATH.read_text()) or "OK")
broken = '''{
  "mcpServers": {
    "notes": {"command": "pyhton3", "args": ["./notes_server.py", "Desktop"], "env": {"PORT": 8080}},
  }
}'''
print("broken (trailing comma):", validate_config(broken))
print("broken (comma fixed):  ", *validate_config(broken.replace("},\n  }", "}\n  }")), sep="\n  ")

### 3. Be the host: read the config, launch every server, list tools
This is what Claude Desktop does at startup: read `mcpServers`, spawn each server as a subprocess with `command`, `args` and `env`, and talk to it over stdio. Like the real app, our host writes **one log file per server** with that server's stderr (`mcp-server-NAME.log`) plus a general `mcp.log`.

In [ ]:
LOGS = work / "logs"
LOGS.mkdir(exist_ok=True)


def host_log(msg: str) -> None:
    with open(LOGS / "mcp.log", "a") as f:
        f.write(f"{time.strftime('%H:%M:%S')} {msg}\n")


def params_for(entry: dict) -> StdioServerParameters:
    return StdioServerParameters(command=entry["command"], args=entry.get("args", []),
                                 env=entry.get("env"), cwd=entry.get("cwd"))


async def connect_all(stack: AsyncExitStack, cfg: dict) -> dict[str, Client]:
    clients = {}
    for name, entry in cfg["mcpServers"].items():
        errlog = stack.enter_context(open(LOGS / f"mcp-server-{name}.log", "a"))
        host_log(f"[{name}] launching: {entry['command']} {' '.join(entry.get('args', []))}")
        try:
            clients[name] = await stack.enter_async_context(Client(stdio_client(params_for(entry), errlog=errlog)))
            host_log(f"[{name}] connected, protocol {clients[name].protocol_version}")
        except Exception as exc:
            host_log(f"[{name}] FAILED: {exc!r}")
    return clients


cfg = json.loads(CONFIG_PATH.read_text())
async with AsyncExitStack() as stack:
    clients = await connect_all(stack, cfg)
    for name, client in clients.items():
        tools = (await client.list_tools()).tools
        print(f"{name}: {[t.name for t in tools]}")
    notes = clients["notes"]
    files = (await notes.call_tool("list_files", {})).structured_content["result"]
    print("list_files ->", [Path(f).name for f in files])
    print("read_file  ->", (await notes.call_tool("read_file", {"path": files[0]})).content[0].text.strip())
    escape = await notes.call_tool("read_file", {"path": str(SERVER)})
    print("outside the allowed dirs ->", escape.is_error, escape.content[0].text)

print("\n--- mcp.log ---\n" + (LOGS / "mcp.log").read_text())
print("--- mcp-server-notes.log ---\n" + (LOGS / "mcp-server-notes.log").read_text())

### 4. What environment does a stdio server actually get?
The debugging guide warns that stdio servers **inherit only a limited subset of environment variables**. The Python SDK makes this concrete: it passes a small allowlist (`HOME`, `LOGNAME`, `PATH`, `SHELL`, `TERM`, `USER` on POSIX), merged with your `env`. A secret exported in your shell does **not** reach the server unless you put it in `env`.

In [ ]:
os.environ["MY_SHELL_SECRET"] = "only-in-the-host"
async with Client(params_for(cfg["mcpServers"]["notes"])) as c:
    seen = (await c.call_tool("visible_env", {})).structured_content["result"]
print("server sees:", seen)
print("MY_SHELL_SECRET visible?", "MY_SHELL_SECRET" in seen, "| NOTES_API_TOKEN visible?", "NOTES_API_TOKEN" in seen)

### 5. How approval works: a consent gate in the host
In Claude Desktop, *every* filesystem operation waits for your approval. The gate belongs to the **host**, not the protocol: before forwarding `tools/call`, the host asks the user. Here a scripted "user" approves reads and denies writes. A denied call never reaches the server.

In [ ]:
def ask_user(server: str, tool: str, args: dict) -> bool:
    decision = not tool.startswith("write")              # scripted user: allow reads, deny writes
    print(f"  [approval] {server}.{tool}({json.dumps(args)[:60]}) -> {'ALLOW' if decision else 'DENY'}")
    return decision


async def guarded_call(client: Client, server: str, tool: str, args: dict):
    if not ask_user(server, tool, args):
        return {"isError": True, "content": "The user declined this tool call."}
    r = await client.call_tool(tool, args)
    return {"isError": r.is_error, "content": r.structured_content or r.content[0].text}


async with Client(params_for(cfg["mcpServers"]["notes"])) as c:
    print(await guarded_call(c, "notes", "read_file", {"path": str(work / "Desktop" / "todo.md")}))
    print(await guarded_call(c, "notes", "write_file", {"path": str(work / "Desktop" / "poem.txt"), "text": "roses"}))
print("poem.txt exists?", (work / "Desktop" / "poem.txt").exists())

### 6. Troubleshooting: reproduce the classic failures
Each case below is a real-world mistake. Watch what the **host** sees (often just a closed connection) versus what the **server log** says. That gap is why the docs send you to `mcp-server-NAME.log` first.

In [ ]:
def root_cause(exc: BaseException) -> BaseException:
    while isinstance(exc, BaseExceptionGroup) and len(exc.exceptions) == 1:
        exc = exc.exceptions[0]
    return exc


good = cfg["mcpServers"]["notes"]
CASES = {
    "relative path": {**good, "args": [str(SERVER), "Desktop"]},
    "missing env var": {k: v for k, v in good.items() if k != "env"},
    "non-existent command": {**good, "command": "pyhton3"},
}
for label, entry in CASES.items():
    log_file = LOGS / f"case-{label.replace(' ', '-')}.log"
    t0 = time.time()
    try:
        with open(log_file, "w") as errlog:
            async with Client(stdio_client(params_for(entry), errlog=errlog)) as c:
                await c.list_tools()
        outcome = "connected?!"
    except BaseException as exc:
        cause = root_cause(exc)
        outcome = f"{type(cause).__name__}: {cause}"
    print(f"### {label}  ({time.time() - t0:.1f}s)\n  host sees : {outcome}")
    print(f"  server log: {log_file.read_text().strip() or '(empty: the process never started)'}\n")

The fixes match the docs' checklist: use **absolute paths** (the server resolved `Desktop` against the host's working directory), put required secrets in **`env`**, and make `command` something the host can find. Run it by hand first, or give an absolute path such as the output of `which uv`.

### 7. A "remote" server over Streamable HTTP, with a bearer token
Remote servers are reached by **URL** instead of a launch command. In Claude this is a Custom Connector, and the client completes authentication (usually OAuth) when you connect. Here a localhost server stands in for the internet. A small ASGI wrapper rejects any request without the right `Authorization` header with `401` and a `WWW-Authenticate` challenge, the way a protected MCP server signals that auth is needed (module 6 covers the real OAuth flow).

In [ ]:
import httpx2
import uvicorn
from mcp.client.streamable_http import streamable_http_client
from mcp.server import MCPServer

remote = MCPServer("tickets")


@remote.tool()
def open_tickets(team: str) -> list[str]:
    """Open tickets for a team."""
    return {"platform": ["PLAT-7 flaky deploy", "PLAT-9 disk alert"]}.get(team, [])


TOKEN = "demo-access-token"
requests_seen = []
mcp_app = remote.streamable_http_app()


async def protected(scope, receive, send):
    if scope["type"] == "http":
        headers = dict(scope["headers"])
        auth = headers.get(b"authorization", b"").decode()
        requests_seen.append((headers.get(b"mcp-method", b"-").decode(), "token" if auth else "no token"))
        if auth != f"Bearer {TOKEN}":
            await send({"type": "http.response.start", "status": 401,
                        "headers": [(b"content-type", b"application/json"),
                                    (b"www-authenticate", b'Bearer realm="tickets"')]})
            await send({"type": "http.response.body", "body": b'{"error": "invalid_token"}'})
            return
    await mcp_app(scope, receive, send)


sock = socket.socket()
sock.bind(("127.0.0.1", 0))
PORT = sock.getsockname()[1]
sock.close()
http_server = uvicorn.Server(uvicorn.Config(protected, host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=http_server.run, daemon=True).start()
while not http_server.started:
    await asyncio.sleep(0.05)
URL = f"http://127.0.0.1:{PORT}/mcp"
print("remote server at", URL)

Connect by URL. To send a custom header, give `streamable_http_client` your own `httpx2.AsyncClient` (the SDK's HTTP library) and pass that transport to `Client`.

In [ ]:
async with httpx2.AsyncClient(headers={"Authorization": f"Bearer {TOKEN}"}, trust_env=False) as http:
    async with Client(streamable_http_client(URL, http_client=http)) as c:
        print("negotiated:", c.protocol_version)
        r = await c.call_tool("open_tickets", {"team": "platform"})
        print("open_tickets ->", r.structured_content)

try:
    async with Client(URL) as c:                       # no Authorization header
        await c.list_tools()
except BaseException as exc:
    print("\nwithout a token ->", type(root_cause(exc)).__name__, root_cause(exc))

async with httpx2.AsyncClient(trust_env=False) as raw:
    resp = await raw.post(URL, json={"jsonrpc": "2.0", "id": 1, "method": "server/discover", "params": {}},
                          headers={"Accept": "application/json, text/event-stream",
                                   "MCP-Protocol-Version": "2026-07-28", "Mcp-Method": "server/discover"})
print("raw HTTP:", resp.status_code, "| WWW-Authenticate:", resp.headers.get("www-authenticate"), "|", resp.text)
print("\nwhat the server saw (Mcp-Method, auth):", requests_seen)

The SDK reports the unauthenticated attempt only as a generic error. The **HTTP status and `WWW-Authenticate` header** are what tell you it is an auth problem, which is why a host's logs (or the Inspector) matter. The server log also shows the client probing `server/discover` and then retrying the legacy handshake (`-`, no `Mcp-Method`) before giving up.

### 8. One config, local and remote
Many tools reuse the `mcpServers` shape and add a remote entry type. The MCP Inspector's catalog, for example, accepts `"type": "http"` with a `url` and `headers`. Claude Desktop's own `claude_desktop_config.json` is documented for **local** launch entries; remote servers are added in Claude as **Custom Connectors** through Settings → Connectors. Below is one host function that connects either kind from such a catalog.

In [ ]:
catalog = {"mcpServers": {
    "notes": cfg["mcpServers"]["notes"],
    "tickets": {"type": "http", "url": URL, "headers": {"Authorization": f"Bearer {TOKEN}"}},
}}


async def open_entry(stack: AsyncExitStack, entry: dict) -> Client:
    if entry.get("type") == "http":
        http = await stack.enter_async_context(httpx2.AsyncClient(headers=entry.get("headers"), trust_env=False))
        return await stack.enter_async_context(Client(streamable_http_client(entry["url"], http_client=http)))
    return await stack.enter_async_context(Client(params_for(entry)))


async with AsyncExitStack() as stack:
    for name, entry in catalog["mcpServers"].items():
        c = await open_entry(stack, entry)
        kind = "remote (Streamable HTTP)" if entry.get("type") == "http" else "local (stdio)"
        print(f"{name:<8} {kind:<26} tools={[t.name for t in (await c.list_tools()).tools]}")

http_server.should_exit = True

### 9. Optional: let Claude use the configured servers
With `ANTHROPIC_API_KEY` set, this is a miniature Claude Desktop: tools from every configured server (namespaced `server__tool`), the consent gate from section 5, and the loop from lesson 5.2.

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    async with AsyncExitStack() as stack:
        clients = await connect_all(stack, cfg)
        tools, route = [], {}
        for server_name, client in clients.items():
            for t in (await client.list_tools()).tools:
                tools.append({"name": f"{server_name}__{t.name}", "description": t.description or "",
                              "input_schema": t.input_schema})
                route[f"{server_name}__{t.name}"] = (server_name, client, t.name)
        messages = [{"role": "user", "content": "What's on my todo list? Then save a two-line poem about it "
                     "to my Desktop as poem.txt."}]
        response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=tools, messages=messages)
        while response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type == "tool_use":
                    server_name, client, tool = route[block.name]
                    out = await guarded_call(client, server_name, tool, block.input)
                    results.append({"type": "tool_result", "tool_use_id": block.id,
                                    "content": json.dumps(out["content"]), "is_error": out["isError"]})
            messages.append({"role": "user", "content": results})
            response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=tools,
                                           messages=messages)
        print("Claude:", "".join(b.text for b in response.content if b.type == "text"))
else:
    print("skipped: set ANTHROPIC_API_KEY to watch Claude read your todo list and get its write request denied.")

### Your turn
1. Add a second server to `config` (for example a copy of `notes_server.py` that serves only `Downloads`). Re-run section 3: are both listed, and does each get its own `mcp-server-NAME.log`?
2. Add `"cwd": str(work)` to the "relative path" case in section 6. Does it start now? Why do the docs still recommend absolute paths over relying on `cwd`?
3. Change the remote wrapper to also accept an `X-API-Key` header, and connect with it. What does a host lose when a server uses static API keys instead of OAuth (think scopes, expiry, revocation)?
4. Make `ask_user` remember "always allow `read_file` for this server" after the first approval. Where should that grant be stored, and when should it expire?